In [1]:
import pandas as pd
import requests
from tqdm import tqdm
from save_asc_to_npy import save_all_asc_to_single_npy
import merge_npy_files as mnf
# reload imports
%load_ext autoreload
%autoreload 2
user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"

In [3]:
links_file = 'links/rgealti_links.txt'
with open(links_file, 'r') as f:
    links = f.read().splitlines()

len(links)

302

In [4]:
# https://data.geopf.fr/telechargement/download/RGEALTI/RGEALTI_2-0_1M_ASC_WGS84UTM20-MART87_D972_2015-10-21/RGEALTI_2-0_1M_ASC_WGS84UTM20-MART87_D972_2015-10-21.7z
# https://data.geopf.fr/telechargement/download/RGEALTI/RGEALTI_2-0_5M_ASC_LAMB93-IGN69_D001_2023-08-08/RGEALTI_2-0_5M_ASC_LAMB93-IGN69_D001_2023-08-08.7z

parsed = []
for link in links:
    parts = link.split('/')[-1].split('_')
    resolution = parts[2][0]
    department = parts[5]
    parsed.append([resolution, department, link])

df_links = pd.DataFrame(parsed, columns=['resolution', 'department', 'link'])
df_links

,resolution,department,link
0,1,D001,https://data.geopf.fr/telechargement/download/...
1,1,D001,https://data.geopf.fr/telechargement/download/...
2,1,D002,https://data.geopf.fr/telechargement/download/...
3,1,D002,https://data.geopf.fr/telechargement/download/...
4,1,D003,https://data.geopf.fr/telechargement/download/...
...,...,...,...
297,5,D093,https://data.geopf.fr/telechargement/download/...
298,5,D094,https://data.geopf.fr/telechargement/download/...
299,5,D095,https://data.geopf.fr/telechargement/download/...
300,5,D02A,https://data.geopf.fr/telechargement/download/...


In [5]:
# python
import os
import glob
import subprocess
from pathlib import Path
from typing import List, Optional

def _merge_parts(parts: List[Path], out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("wb") as fout:
        for p in parts:
            with p.open("rb") as fin:
                while True:
                    chunk = fin.read(1024 * 1024)
                    if not chunk:
                        break
                    fout.write(chunk)

def _extract_with_py7zr(archive: Path, out_dir: Path) -> None:
    try:
        import py7zr
        with py7zr.SevenZipFile(str(archive), mode="r") as z:
            z.extractall(path=str(out_dir))
        return
    except Exception as e:
        raise RuntimeError(f"py7zr failed: {e}")

def _extract_with_7z(archive: Path, out_dir: Path) -> None:
    # Attends que 7z (7z.exe) soit dans le PATH sur Windows
    subprocess.run(["7z", "x", str(archive), f"-o{str(out_dir)}", "-y"], check=True)

def extract_archive_from_folder(folder: str, out_dir: Optional[str] = None) -> Path:
    """
    Détecte et extrait une archive 7z ou ses parties dans `folder`.
    Retourne le chemin du dossier d'extraction.
    """
    folder_p = Path(folder)
    if out_dir:
        out_p = Path(out_dir)
    else:
        out_p = folder_p / "extracted"
    out_p.mkdir(parents=True, exist_ok=True)

    # 1) Cherche un fichier .7z complet
    sevenz = sorted(folder_p.glob("*.7z"))
    if sevenz:
        archive = sevenz[0]
        try:
            _extract_with_py7zr(archive, out_p)
            return out_p
        except Exception:
            _extract_with_7z(archive, out_p)
            return out_p

    # 2) Cherche .7z.001 (nom complet comme name.7z.001)
    parts_7z001 = sorted(folder_p.glob("*.7z.001"))
    if parts_7z001:
        first = parts_7z001[0]
        try:
            _extract_with_7z(first, out_p)
            return out_p
        except subprocess.CalledProcessError:
            # fallback: merge puis extract
            merged = folder_p / "merged.7z"
            all_parts = sorted(folder_p.glob(f"{first.stem.split('.7z')[0]}*.7z.*")) or parts_7z001
            _merge_parts(all_parts, merged)
            try:
                _extract_with_py7zr(merged, out_p)
            except Exception:
                _extract_with_7z(merged, out_p)
            finally:
                if merged.exists():
                    merged.unlink()
            return out_p

    # 3) Cherche parties generiques .001/.002...
    parts_001 = sorted(folder_p.glob("*.001"))
    if parts_001:
        try:
            _extract_with_7z(parts_001[0], out_p)
            return out_p
        except subprocess.CalledProcessError:
            merged = folder_p / "merged.7z"
            _merge_parts(parts_001, merged)
            try:
                _extract_with_py7zr(merged, out_p)
            except Exception:
                _extract_with_7z(merged, out_p)
            finally:
                if merged.exists():
                    merged.unlink()
            return out_p

    raise FileNotFoundError(f"Aucune archive 7z ni parties trouvées dans ` {folder} `")


In [6]:
def download_with_ua_progress(url: str, out_dir: str, user_agent: str = None, timeout: int = 30, chunk_size: int = 8192) -> str:
    os.makedirs(out_dir, exist_ok=True)
    local_path = os.path.join(out_dir, os.path.basename(url))
    if os.path.exists(local_path):
        print(f'File {local_path} already exists, skipping download.')
        return local_path
    headers = {'User-Agent': user_agent or "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}
    with requests.get(url, headers=headers, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        total = int(r.headers.get('content-length') or 0)
        total_for_tqdm = total if total > 0 else None
        with open(local_path, 'wb') as f, tqdm(total=total_for_tqdm, unit='B', unit_scale=True, unit_divisor=1024, desc=os.path.basename(local_path)) as bar:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if not chunk:
                    continue
                f.write(chunk)
                bar.update(len(chunk))
    return local_path

In [7]:
resolution = '5'
#department = 'D074'
departments = df_links[df_links['resolution'] == resolution]['department'].unique().tolist()


In [ ]:
# download all 1m files
for department in departments:
    df_subset = df_links[(df_links['resolution'] == resolution) & (df_links['department'] == department)]
    for idx, row in df_subset.iterrows():
        link = row['link']
        print(f'Downloading {link}...')

        output_dir = f'data/rgealti/{resolution}m/{department}'
        if os.path.exists(output_dir):
            print(f'Directory {output_dir} already exists, skipping download and extraction.')
            continue
        os.makedirs(output_dir, exist_ok=True)

        download_with_ua_progress(link, output_dir, user_agent)
        extract_archive_from_folder(output_dir, output_dir)

RGEALTI_2-0_5M_ASC_LAMB93-IGN69_D001_2023-08-08.7z:  37%|███▋      | 164M/444M [00:54<01:33, 3.14MB/s] 

In [ ]:
print('Combining all departments into a single npz file...')
data = mnf.merge_npy_files(input_dir='npy',output_path='D:\\all_departments.npy')

In [ ]:
data

In [ ]:
from compute import BatchProcessor
from datetime import datetime

processor = BatchProcessor()

daylight = processor.process_from_npz(
    input_npz=data,
    output_npz='D:\\daylight_results.npz',
    date=datetime(2024, 6, 21),  # Summer solstice
    batch_size=240,               # Adjust for your GPU memory
    pixel_size=5.0,               # 5 meters for 5M resolution data
    lat_center=46.0,              # France center latitude
    lon_center=2.0                # France center longitude
)

print(f'Computed daylight for {len(daylight)} tiles')

In [ ]:
python3 compute_daylight_npz.py D:\\all_departments.npy D:\\daylight_results.npz --chunk-size 500